# 🗄️ CLASE 3 — SQL para la Exploración de Datos (Data Discovery)
### Master IA · Dev Senior Code · Módulo 1

---

## ¿Por qué SQL en un curso de IA?

Antes de entrenar cualquier modelo de Machine Learning, necesitas datos. Y en el mundo real, **los datos no viven en archivos CSV perfectamente ordenados** — viven en bases de datos relacionales dentro de empresas, startups, bancos, hospitales y gobiernos.

Según múltiples encuestas de la industria (Kaggle, Stack Overflow), el **80% del tiempo** de un Data Scientist o ML Engineer se gasta en obtener, limpiar y entender datos. SQL es la herramienta con la que accedes a esa información.

> **SQL no es opcional en IA — es el primer paso antes de cualquier modelo.**

### ¿Dónde usarás SQL en tu carrera de IA?

- **Exploración de datos (Data Discovery):** entender qué hay en la base de datos antes de modelar
- **Feature Engineering:** construir las variables que va a consumir tu modelo
- **ETL (Extract, Transform, Load):** mover y transformar datos entre sistemas
- **Monitoreo de modelos:** consultar métricas de producción almacenadas en BD
- **Reportes de negocio:** responder preguntas del equipo comercial sin esperar a un analista

---

## 🗂️ ¿Qué es una Base de Datos Relacional?

Una **base de datos relacional** organiza la información en **tablas** (como hojas de cálculo) que se relacionan entre sí mediante columnas comunes llamadas **claves**.

Cada tabla tiene:
- **Filas (registros):** cada fila es un elemento del mundo real (un cliente, una venta, un producto)
- **Columnas (campos):** cada columna es una característica de ese elemento (nombre, precio, fecha)
- **Clave primaria (PRIMARY KEY):** columna que identifica de forma única cada fila
- **Clave foránea (FOREIGN KEY):** columna que referencia la clave primaria de otra tabla

### ¿Por qué no usar solo Excel o Pandas?

| Situación | Excel/Pandas | SQL en BD |
|-----------|-------------|------------|
| 10.000 filas | ✅ Perfecto | ✅ También |
| 10 millones de filas | ❌ Se cuelga | ✅ Sin problema |
| Múltiples usuarios a la vez | ❌ Conflictos | ✅ Diseñado para eso |
| Datos actualizándose en tiempo real | ❌ Snapshot | ✅ Siempre al día |
| Relacionar 5 tablas distintas | 😰 Muy manual | ✅ Un solo JOIN |

---

## 🏗️ El Modelo de Datos que usaremos hoy

Vamos a trabajar con la base de datos de un **e-commerce colombiano**. Tiene 3 tablas:

```
┌─────────────────┐        ┌──────────────────┐        ┌──────────────────┐
│    CLIENTES     │        │     VENTAS       │        │    PRODUCTOS     │
├─────────────────┤        ├──────────────────┤        ├──────────────────┤
│ id_cliente  🔑  │◄──┐    │ id_venta    🔑   │   ┌───►│ id_producto  🔑  │
│ nombre          │   └────│ id_cliente  🔗   │   │    │ nombre           │
│ ciudad          │        │ id_producto 🔗   │───┘    │ categoria        │
│ categoria       │        │ cantidad         │        │ precio           │
│ edad            │        │ fecha            │        │ stock            │
│ activo          │        │ total            │        └──────────────────┘
└─────────────────┘        └──────────────────┘

🔑 = Clave primaria (PRIMARY KEY) — identifica únicamente cada fila
🔗 = Clave foránea (FOREIGN KEY)  — referencia a otra tabla
```

**Cómo leer el diagrama:**
- Un **cliente** puede tener **muchas ventas** (relación 1 a muchos)
- Un **producto** puede aparecer en **muchas ventas** (relación 1 a muchos)
- La tabla **ventas** es la tabla central (tabla de hechos) que conecta clientes con productos

---

## 📋 Contenido de la clase

| # | Bloque | Tiempo |
|---|--------|---------|
| 0 | Setup: creamos la base de datos juntos | 15 min |
| 1 | SELECT, DISTINCT y alias | 25 min |
| 2 | WHERE, IN, BETWEEN y operadores lógicos | 25 min |
| 3 | GROUP BY, HAVING y funciones de agregación | 30 min |
| 4 | JOINs: cruzar tablas | 25 min |
| 5 | Ejercicio real: reporte ejecutivo de ventas | 30 min |

---
## ⚙️ BLOQUE 0 — Setup: Creamos la base de datos juntos

### ¿Qué motor de base de datos usamos?

Usaremos **SQLite**, que viene incluido en Python sin instalar nada. Es perfecto para aprender porque corre completamente en memoria.

En la industria usarías:
- **PostgreSQL** → el estándar en startups y proyectos de IA
- **MySQL / MariaDB** → muy común en aplicaciones web
- **BigQuery** (Google) / **Redshift** (AWS) → para datos a escala masiva
- **SQL Server** → empresas corporativas / Microsoft

La buena noticia: **el SQL que aprenderás hoy funciona en todos ellos con cambios mínimos.**

### ¿Qué es CREATE TABLE?

Es la instrucción que **define la estructura** de una tabla: qué columnas tiene y qué tipo de dato guarda cada una. Es como crear el encabezado de una hoja de cálculo antes de meter datos.

Tipos de datos más comunes en SQL:
- `INTEGER` → números enteros (1, 2, 100)
- `REAL` → números decimales (3.14, 1500000.50)
- `TEXT` → cadenas de texto ('Bogotá', 'Ana Torres')
- `DATE` / `TEXT` → fechas (SQLite las guarda como texto 'YYYY-MM-DD')

In [ ]:
# ── Importar librerías necesarias ───────────────────────────────────────────
import sqlite3  # Importamos sqlite3, que ya viene incluido con Python.

# ── Crear conexión a la base de datos en memoria ────────────────────────────
# sqlite3.connect() abre una conexión nueva a una base de datos SQLite.
# ':memory:' indica que la base de datos vive en RAM y no en un archivo físico.
# Esto permite que la clase arranque limpia cada vez que ejecutamos el notebook.
# En producción usaríamos un archivo, por ejemplo: sqlite3.connect('mi_base.db').
conn = sqlite3.connect(':memory:')  # Guardamos la conexión para reutilizarla en todas las consultas.

# ── Crear cursor para ejecutar consultas SQL ────────────────────────────────
cursor = conn.cursor()  # Creamos el cursor que enviará instrucciones SQL a la base de datos.

print('✅ Conexión a SQLite establecida correctamente')  # Confirmamos que la conexión fue creada.
print('   La base de datos vive en memoria RAM')  # Recordamos dónde está guardada la base de datos.
print('   Motor: SQLite 3 (incluido en Python, sin instalación adicional)')  # Indicamos el motor usado.

### Creando las tablas: entendiendo CREATE TABLE e INSERT

Vamos a crear las 3 tablas de nuestro e-commerce. Presta atención a:
- **PRIMARY KEY:** le dice a la BD que esa columna identifica únicamente cada fila. No puede repetirse ni ser nula.
- **Tipos de datos:** cada columna almacena solo un tipo de información
- **INSERT INTO:** inserta filas de datos en la tabla ya creada. Los valores van en el mismo orden que las columnas definidas en el CREATE.

In [ ]:
# ── executescript() ejecuta múltiples instrucciones SQL de una vez ───────────
# Usamos triple comillas """ para escribir SQL en varias líneas (más legible)
# Los comentarios dentro del SQL van con doble guión: --

conn.executescript("""

-- ═══════════════════════════════════════════════════════
--  TABLA 1: CLIENTES
--  Almacena la información de cada cliente de la tienda
-- ═══════════════════════════════════════════════════════
CREATE TABLE clientes (
    id_cliente   INTEGER PRIMARY KEY,  -- identificador único, no puede repetirse
    nombre       TEXT,                 -- nombre completo del cliente
    ciudad       TEXT,                 -- ciudad de residencia
    categoria    TEXT,                 -- segmento: 'regular', 'premium', 'vip'
    edad         INTEGER,              -- edad en años
    activo       INTEGER               -- estado: 1=activo, 0=inactivo (baja)
);

-- ═══════════════════════════════════════════════════════
--  TABLA 2: PRODUCTOS
--  Catálogo de productos disponibles en la tienda
-- ═══════════════════════════════════════════════════════
CREATE TABLE productos (
    id_producto  INTEGER PRIMARY KEY,  -- identificador único del producto
    nombre       TEXT,                 -- nombre del producto
    categoria    TEXT,                 -- categoría: 'Tecnología', 'Muebles', etc.
    precio       REAL,                 -- precio unitario en pesos colombianos
    stock        INTEGER               -- unidades disponibles en inventario
);

-- ═══════════════════════════════════════════════════════
--  TABLA 3: VENTAS
--  Registra cada transacción realizada
--  Es la tabla central que une clientes con productos
-- ═══════════════════════════════════════════════════════
CREATE TABLE ventas (
    id_venta     INTEGER PRIMARY KEY,  -- identificador único de la transacción
    id_cliente   INTEGER,              -- FK → referencia a clientes.id_cliente
    id_producto  INTEGER,              -- FK → referencia a productos.id_producto
    cantidad     INTEGER,              -- cuántas unidades se compraron
    fecha        TEXT,                 -- fecha de la venta (formato YYYY-MM-DD)
    total        REAL                  -- valor total de la transacción en pesos
);

-- ═══════════════════════════════════════════════════════
--  INSERTAR DATOS EN CLIENTES
--  Sintaxis: INSERT INTO tabla VALUES (val1, val2, ..., valN)
--  Los valores van en el mismo orden que las columnas del CREATE TABLE
-- ═══════════════════════════════════════════════════════
INSERT INTO clientes VALUES
 (1,  'Ana Torres',    'Bogotá',       'vip',     34, 1),
 (2,  'Luis Gómez',    'Medellín',     'premium', 28, 1),
 (3,  'Sara Ríos',     'Cali',         'regular', 45, 1),
 (4,  'Pedro Díaz',    'Bogotá',       'regular', 22, 0),  -- cliente inactivo
 (5,  'María López',   'Barranquilla', 'premium', 31, 1),
 (6,  'Carlos Mora',   'Medellín',     'vip',     52, 1),
 (7,  'Julia Soto',    'Bogotá',       'regular', 27, 1),
 (8,  'Tomás Vera',    'Cali',         'regular', 19, 0),  -- cliente inactivo
 (9,  'Camila Ruiz',   'Bogotá',       'premium', 38, 1),
 (10, 'Diego Mora',    'Medellín',     'vip',     44, 1);

-- ═══════════════════════════════════════════════════════
--  INSERTAR DATOS EN PRODUCTOS
-- ═══════════════════════════════════════════════════════
INSERT INTO productos VALUES
 (1,  'Laptop Pro',          'Tecnología', 3500000, 15),
 (2,  'Mouse Inalámbrico',   'Tecnología',   85000, 80),
 (3,  'Teclado RGB',         'Tecnología',  220000, 45),
 (4,  'Monitor 27"',         'Tecnología', 1200000, 20),
 (5,  'Silla Ergonómica',    'Muebles',     890000, 12),
 (6,  'Escritorio',          'Muebles',     650000,  8),
 (7,  'Audífonos BT',        'Tecnología',  310000, 35),
 (8,  'Webcam HD',           'Tecnología',  195000, 50),
 (9,  'Lámpara LED',         'Hogar',        75000,100),
 (10, 'Cuaderno A5',         'Papelería',    12000,200);

-- ═══════════════════════════════════════════════════════
--  INSERTAR DATOS EN VENTAS
--  Nota: id_cliente e id_producto deben existir en sus tablas
-- ═══════════════════════════════════════════════════════
INSERT INTO ventas VALUES
 (1,  1, 1, 1, '2024-01-15', 3500000),
 (2,  1, 2, 2, '2024-01-15',  170000),
 (3,  2, 3, 1, '2024-01-20',  220000),
 (4,  3, 9, 3, '2024-01-22',  225000),
 (5,  2, 7, 1, '2024-02-01',  310000),
 (6,  5, 4, 1, '2024-02-10', 1200000),
 (7,  6, 1, 2, '2024-02-14', 7000000),
 (8,  4,10, 5, '2024-02-20',   60000),
 (9,  9, 5, 1, '2024-03-05',  890000),
 (10, 7, 8, 1, '2024-03-10',  195000),
 (11,10, 6, 1, '2024-03-15',  650000),
 (12, 1, 4, 1, '2024-03-18', 1200000),
 (13, 3, 2, 3, '2024-03-20',  255000),
 (14, 6, 7, 2, '2024-04-02',  620000),
 (15, 2, 8, 1, '2024-04-10',  195000);

""")

print('✅ Base de datos creada y cargada exitosamente:')
print('   📋 clientes  — 10 registros (8 activos, 2 inactivos)')
print('   📦 productos — 10 registros (4 categorías diferentes)')
print('   🛒 ventas    — 15 transacciones (enero a abril 2024)')

In [ ]:
# ── Verificar que las tablas existen y tienen datos ─────────────────────────
# En SQLite, sqlite_master es una tabla interna que guarda el catálogo
# de objetos de la base de datos (tablas, índices, vistas...)
# Esta consulta es equivalente a \dt en PostgreSQL o SHOW TABLES en MySQL

print('Tablas disponibles en la base de datos:')  # Mostramos un título antes del resultado.
# Ejecutamos la consulta SQL directamente con el cursor.
cursor.execute("""
    SELECT name AS tabla, type AS tipo
    FROM   sqlite_master
    WHERE  type = 'table'
    ORDER  BY name
""")
filas = cursor.fetchall()  # Guardamos todas las filas que devolvió la consulta.

for fila in filas:  # Recorremos cada fila obtenida desde SQLite.
    print(fila)  # Imprimimos la fila completa como una tupla.

---
## 📋 BLOQUE 1 — SELECT, DISTINCT y Alias

### La estructura base de cualquier consulta SQL

Todo en SQL empieza con `SELECT`. Es la instrucción con la que le **preguntas** cosas a la base de datos.

```sql
SELECT  columna1, columna2, ...   -- ¿Qué quiero ver? (las columnas)
FROM    nombre_tabla              -- ¿De dónde? (la tabla fuente)
WHERE   condición                 -- ¿Con qué filtro? (opcional)
ORDER BY columna ASC/DESC         -- ¿En qué orden? (opcional)
LIMIT   numero                    -- ¿Cuántas filas máximo? (opcional)
```

**Importante:** SQL no distingue mayúsculas de minúsculas en las palabras clave (`SELECT` = `select` = `Select`). Por convención profesional se escriben en **MAYÚSCULAS** para diferenciarlas de los nombres de tablas y columnas.

### ¿Qué haremos en este bloque?
1. Ver todos los datos de una tabla con `SELECT *`
2. Seleccionar solo las columnas que necesitamos
3. Renombrar columnas con `AS` (alias)
4. Crear columnas calculadas
5. Eliminar duplicados con `DISTINCT`
6. Ordenar y limitar resultados

In [ ]:
# ── SELECT * : traer TODAS las columnas de la tabla ─────────────────────────
#
# El asterisco (*) es un comodín que significa "todas las columnas"
# Es muy útil para explorar una tabla que no conoces
#
# ⚠️ ADVERTENCIA PROFESIONAL: Evita SELECT * en código de producción porque:
#    1. Trae datos que quizás no necesitas (más lento en redes y BD grandes)
#    2. Si alguien agrega o elimina columnas, tu código puede romperse
#    3. No queda claro qué datos estás usando realmente
#
# Úsalo solo para exploración inicial y aprendizaje

cursor.execute("SELECT * FROM clientes")  # Ejecutamos la consulta directamente en SQLite.
filas = cursor.fetchall()  # Traemos todos los clientes encontrados por la consulta.

for fila in filas:  # Recorremos cada cliente devuelto por fetchall().
    print(fila)  # Imprimimos la fila completa como una tupla.

In [ ]:
# ── SELECT de columnas específicas ──────────────────────────────────────────
#
# La práctica correcta: pedir solo las columnas que vas a usar
# Esto es más eficiente, más legible y más seguro en producción
# El orden en que listas las columnas es el orden en que aparecen en el resultado

# Ejecutamos la consulta directamente con sqlite3.
cursor.execute("""
    SELECT nombre,      -- primera columna del resultado
           ciudad,      -- segunda columna
           categoria,   -- tercera columna
           activo       -- cuarta columna: 1=activo, 0=inactivo
    FROM   clientes
""")
filas = cursor.fetchall()  # Guardamos todas las filas resultantes.

for fila in filas:  # Recorremos cada fila de la consulta.
    print(fila)  # Imprimimos nombre, ciudad, categoría y estado activo.

### Alias con AS — Renombrar columnas para más claridad

`AS` le asigna un nombre temporal a una columna **solo en el resultado de la consulta**. No modifica la tabla original.

¿Cuándo usarlo?
- Cuando el nombre de la columna en la BD es críptico o técnico (`id_cli` → `cliente`)
- Cuando creas columnas calculadas (el resultado de `precio * cantidad` necesita un nombre)
- Cuando haces JOINs y dos tablas tienen columnas con el mismo nombre
- Para que los reportes sean legibles por personas no técnicas

In [ ]:
# ── ALIAS con AS ─────────────────────────────────────────────────────────────
#
# Sintaxis: nombre_columna AS nombre_alias
# El alias puede ir con o sin comillas. Si tiene espacios, necesita comillas: AS 'Nombre Completo'

# Ejecutamos la consulta SQL con alias y columna calculada.
cursor.execute("""
    SELECT
        nombre      AS cliente,          -- renombramos 'nombre' a 'cliente'
        ciudad      AS ubicacion,        -- renombramos 'ciudad' a 'ubicacion'
        categoria   AS segmento,         -- renombramos 'categoria' a 'segmento'
        edad,                            -- esta columna queda con su nombre original
        edad * 12   AS edad_en_meses,    -- columna calculada: multiplicación
        activo      AS esta_activo       -- 1 o 0 con nombre más descriptivo
    FROM clientes
""")
filas = cursor.fetchall()  # Traemos todos los registros generados por la consulta.

for fila in filas:  # Recorremos cada resultado.
    print(fila)  # Imprimimos la fila con los alias aplicados.

In [ ]:
# ── ORDER BY: ordenar el resultado ──────────────────────────────────────────
#
# ORDER BY columna ASC  → orden ascendente (A→Z, menor a mayor) - es el defecto
# ORDER BY columna DESC → orden descendente (Z→A, mayor a menor)
# Puedes ordenar por múltiples columnas: ORDER BY ciudad ASC, edad DESC
# El ordenamiento ocurre al final, después de todos los filtros

# Ver los 5 clientes más jóvenes, mostrando solo info relevante
# Ejecutamos la consulta ordenada por edad.
cursor.execute("""
    SELECT
        nombre   AS cliente,
        ciudad,
        edad,
        categoria AS segmento
    FROM   clientes
    ORDER  BY edad ASC     -- de menor a mayor edad
    LIMIT  5               -- mostrar solo los 5 primeros resultados
""")
filas = cursor.fetchall()  # Obtenemos los 5 clientes más jóvenes.

for fila in filas:  # Recorremos los resultados uno por uno.
    print(fila)  # Imprimimos cada cliente encontrado.

### DISTINCT — Eliminar valores duplicados

`DISTINCT` filtra el resultado para mostrar **solo valores únicos**. Es equivalente a `df['columna'].unique()` en pandas.

Es muy usado al explorar una tabla nueva para responder preguntas como:
- ¿Cuántas ciudades distintas tenemos en la base?
- ¿Qué categorías de productos existen?
- ¿Qué estados posibles puede tener un pedido?

Cuando usas `DISTINCT` con múltiples columnas, devuelve las **combinaciones únicas** de esas columnas.

In [ ]:
# ── DISTINCT en una sola columna ─────────────────────────────────────────────
#
# Sin DISTINCT: veríamos 'Bogotá' repetida 4 veces (hay 4 clientes de Bogotá)
# Con DISTINCT: cada ciudad aparece una sola vez
#
# Pregunta de exploración: ¿En qué ciudades tenemos clientes?

print('Ciudades únicas en la base de datos:')  # Mostramos un título antes de imprimir las ciudades.
# Ejecutamos la consulta para traer ciudades sin repetir.
cursor.execute("""
    SELECT DISTINCT ciudad
    FROM   clientes
    ORDER  BY ciudad ASC    -- ordenamos alfabéticamente para leer mejor
""")
filas = cursor.fetchall()  # Obtenemos todas las ciudades únicas.

for fila in filas:  # Recorremos cada ciudad devuelta por SQLite.
    print(fila)  # Imprimimos la ciudad como tupla.

In [ ]:
# ── DISTINCT con múltiples columnas ──────────────────────────────────────────
#
# Aquí DISTINCT trabaja sobre la COMBINACIÓN de ciudad + categoria
# Una fila aparece si esa combinación específica existe en la tabla
#
# Pregunta de exploración: ¿Qué combinaciones de ciudad y segmento tenemos?
# Muy útil para detectar si todas las ciudades tienen todos los segmentos

# Ejecutamos la consulta para traer combinaciones únicas.
cursor.execute("""
    SELECT DISTINCT
        ciudad,
        categoria AS segmento
    FROM   clientes
    ORDER  BY ciudad, categoria    -- doble ordenamiento: primero ciudad, luego segmento
""")
filas = cursor.fetchall()  # Guardamos todas las combinaciones ciudad-segmento.

for fila in filas:  # Recorremos cada combinación encontrada.
    print(fila)  # Imprimimos ciudad y segmento.

---
## 🔍 BLOQUE 2 — Filtrado y Lógica: WHERE, IN, BETWEEN, LIKE

### ¿Qué es WHERE?

`WHERE` es la cláusula que **filtra filas** según una condición. Solo las filas donde la condición es verdadera (`TRUE`) pasan al resultado.

Es equivalente a `.query()` o `.loc[condición]` en pandas.

### Operadores disponibles

| Operador | Significado | Ejemplo |
|----------|-------------|--------|
| `=` | Igual | `ciudad = 'Bogotá'` |
| `!=` o `<>` | Diferente | `categoria != 'regular'` |
| `>` | Mayor que | `edad > 30` |
| `>=` | Mayor o igual | `edad >= 18` |
| `<` | Menor que | `precio < 500000` |
| `<=` | Menor o igual | `stock <= 10` |
| `BETWEEN a AND b` | Rango inclusivo | `edad BETWEEN 25 AND 40` |
| `IN (v1, v2)` | En una lista | `ciudad IN ('Bogotá', 'Cali')` |
| `NOT IN (v1, v2)` | No está en lista | `categoria NOT IN ('regular')` |
| `LIKE 'patrón'` | Búsqueda de texto | `nombre LIKE 'Ana%'` |
| `IS NULL` | Es nulo | `telefono IS NULL` |
| `IS NOT NULL` | No es nulo | `email IS NOT NULL` |

In [ ]:
# ── WHERE con condición simple ───────────────────────────────────────────────
#
# Filtramos solo las filas donde ciudad = 'Bogotá'
# El texto en SQL siempre va entre comillas SIMPLES: 'texto'
# Nunca dobles: "texto" (eso es para nombres de columnas con espacios)
#
# Pregunta de negocio: ¿Cuáles son nuestros clientes en Bogotá?

# Ejecutamos la consulta con el filtro WHERE.
cursor.execute("""
    SELECT nombre, ciudad, categoria, activo
    FROM   clientes
    WHERE  ciudad = 'Bogotá'    -- solo las filas donde ciudad sea exactamente 'Bogotá'
""")
filas = cursor.fetchall()  # Traemos todos los clientes que viven en Bogotá.

for fila in filas:  # Recorremos los clientes encontrados.
    print(fila)  # Imprimimos nombre, ciudad, categoría y estado.

In [ ]:
# ── AND / OR / NOT — Combinar múltiples condiciones ──────────────────────────
#
# AND → AMBAS condiciones deben ser verdaderas para incluir la fila
# OR  → al menos UNA condición debe ser verdadera
# NOT → invierte el resultado de la condición
#
# ⚠️ IMPORTANTE: AND tiene precedencia sobre OR (igual que × sobre + en matemáticas)
# Siempre usa paréntesis para asegurarte del orden de evaluación
#
# Pregunta de negocio: clientes activos mayores de 30 años

# Ejecutamos la consulta con dos condiciones combinadas con AND.
cursor.execute("""
    SELECT nombre, ciudad, edad, categoria
    FROM   clientes
    WHERE  edad > 30          -- condición 1: mayor de 30
    AND    activo = 1         -- condición 2 (AND): además debe estar activo
    ORDER  BY edad DESC
""")
filas = cursor.fetchall()  # Traemos los clientes activos mayores de 30 años.

for fila in filas:  # Recorremos cada cliente del resultado.
    print(fila)  # Imprimimos nombre, ciudad, edad y categoría.

In [ ]:
# ── BETWEEN — Filtrar por un rango de valores ────────────────────────────────
#
# BETWEEN a AND b es equivalente a: >= a AND <= b
# Es INCLUSIVO en ambos extremos: incluye los valores a y b
# Funciona con números, fechas y texto (orden alfabético)
#
# Pregunta de negocio: clientes en el rango de edad productivo (25-40 años)

# Ejecutamos la consulta filtrando con BETWEEN.
cursor.execute("""
    SELECT nombre, edad, ciudad, categoria
    FROM   clientes
    WHERE  edad BETWEEN 25 AND 40    -- incluye 25 y 40 (ambos extremos)
    ORDER  BY edad
""")
filas = cursor.fetchall()  # Obtenemos los clientes cuya edad está en el rango.

for fila in filas:  # Recorremos cada cliente encontrado.
    print(fila)  # Imprimimos nombre, edad, ciudad y categoría.

In [ ]:
# ── Filtrar ventas por rango de fechas con BETWEEN ───────────────────────────
#
# En SQLite las fechas se guardan como texto 'YYYY-MM-DD'
# El formato YYYY-MM-DD permite comparación alfabética directa
# (funciona perfectamente porque '2024-02' < '2024-03' alfabéticamente)
#
# Pregunta de negocio: ventas del primer trimestre (enero y febrero 2024)

# Ejecutamos la consulta filtrando ventas por fecha.
cursor.execute("""
    SELECT id_venta, id_cliente, fecha, total
    FROM   ventas
    WHERE  fecha BETWEEN '2024-01-01' AND '2024-02-28'  -- Q1: enero-febrero
    ORDER  BY fecha
""")
filas = cursor.fetchall()  # Guardamos las ventas que caen dentro del rango.

for fila in filas:  # Recorremos cada venta encontrada.
    print(fila)  # Imprimimos id, cliente, fecha y total.

In [ ]:
# ── IN — Comparar contra una lista de valores ────────────────────────────────
#
# IN (v1, v2, v3) es equivalente a: = v1 OR = v2 OR = v3
# Pero IN es mucho más legible cuando tienes 3 o más opciones
# NOT IN excluye todos los valores de la lista
#
# Pregunta de negocio: clientes VIP y Premium activos en ciudades principales

# Ejecutamos la consulta usando IN para listas de valores.
cursor.execute("""
    SELECT nombre, ciudad, categoria, edad
    FROM   clientes
    WHERE  ciudad    IN ('Bogotá', 'Medellín')    -- solo estas dos ciudades
    AND    categoria IN ('vip', 'premium')        -- solo estos dos segmentos
    AND    activo = 1                             -- además deben estar activos
    ORDER  BY categoria, nombre
""")
filas = cursor.fetchall()  # Guardamos los clientes que cumplen todas las condiciones.

for fila in filas:  # Recorremos cada cliente filtrado.
    print(fila)  # Imprimimos nombre, ciudad, categoría y edad.

In [ ]:
# ── LIKE — Búsqueda de patrones en texto ─────────────────────────────────────
#
# LIKE permite buscar texto con comodines:
#   %  → cualquier secuencia de cero o más caracteres (como * en un buscador)
#   _  → exactamente un carácter cualquiera
#
# Ejemplos de patrones:
#   LIKE 'Ana%'   → empieza con 'Ana': 'Ana', 'Análisis', 'Anabel'
#   LIKE '%Mora'  → termina en 'Mora': 'Carlos Mora', 'Diego Mora'
#   LIKE '%or%'   → contiene 'or': 'Torres', 'Mora', 'Carlos'
#   LIKE 'A___'   → empieza con A y tiene 4 caracteres en total
#
# ⚠️ LIKE es sensible a mayúsculas en algunos motores (en SQLite no)

print('Clientes cuyo apellido contiene "Mora" o "Ríos":')  # Mostramos el objetivo del filtro.
# Ejecutamos la consulta usando LIKE para buscar texto.
cursor.execute("""
    SELECT nombre, ciudad, categoria
    FROM   clientes
    WHERE  nombre LIKE '%Mora%'    -- el nombre contiene 'Mora' en cualquier posición
    OR     nombre LIKE '%Ríos%'    -- O el nombre contiene 'Ríos'
""")
filas = cursor.fetchall()  # Guardamos los clientes que coinciden con el patrón.

for fila in filas:  # Recorremos cada coincidencia.
    print(fila)  # Imprimimos nombre, ciudad y categoría.

In [ ]:
# ── Combinando todo: AND + OR + BETWEEN + IN ─────────────────────────────────
#
# Los paréntesis son CRUCIALES para controlar el orden de evaluación
# Sin paréntesis, AND se evalúa antes que OR (como × antes que + en álgebra)
#
# Pregunta de negocio: clientes de alto valor que vale la pena contactar
# Criterio: (VIP en cualquier ciudad) O (Premium en Bogotá o Medellín)
#           Y que tengan entre 25 y 55 años Y estén activos

# Ejecutamos una consulta con varias condiciones combinadas.
cursor.execute("""
    SELECT nombre, ciudad, categoria, edad
    FROM   clientes
    WHERE  (
               categoria = 'vip'                              -- opción A: cualquier VIP
               OR
               (categoria = 'premium'                         -- opción B: premium...
                AND ciudad IN ('Bogotá', 'Medellín'))         --   ...en ciudades clave
           )
    AND    edad   BETWEEN 25 AND 55    -- condición adicional: rango de edad
    AND    activo = 1                  -- condición adicional: debe estar activo
    ORDER  BY categoria, ciudad
""")
filas = cursor.fetchall()  # Guardamos los clientes que cumplen el criterio de alto valor.

for fila in filas:  # Recorremos cada cliente seleccionado.
    print(fila)  # Imprimimos nombre, ciudad, categoría y edad.

---
## 📊 BLOQUE 3 — Agregaciones: GROUP BY, HAVING y funciones

### ¿Qué son las funciones de agregación?

Las funciones de agregación **calculan un valor resumen** a partir de múltiples filas. En lugar de devolver cada fila individualmente, devuelven **un único valor por grupo**.

Son el equivalente SQL de `.groupby().agg()` en pandas.

| Función | ¿Qué calcula? | Ejemplo de uso |
|---------|---------------|----------------|
| `COUNT(*)` | Número de filas | Total de ventas |
| `COUNT(col)` | Filas no nulas en esa columna | Clientes con email registrado |
| `SUM(col)` | Suma de los valores | Total de ingresos |
| `AVG(col)` | Promedio | Edad promedio de clientes |
| `MIN(col)` | Valor mínimo | Venta más pequeña |
| `MAX(col)` | Valor máximo | Producto más caro |
| `ROUND(val, n)` | Redondea a n decimales | Precio redondeado |

### La regla de oro del GROUP BY

> **Toda columna en el SELECT que NO sea una función de agregación, DEBE estar en el GROUP BY.**

Esto es porque GROUP BY colapsa múltiples filas en una. SQL necesita saber cómo resolver el valor de las columnas no agregadas: la respuesta es que deben ser el criterio de agrupación.

### Diferencia clave: WHERE vs HAVING

```
WHERE  → filtra FILAS    (antes de agrupar)  → NO puede usar funciones de agregación
HAVING → filtra GRUPOS   (después de agrupar) → SÍ puede usar funciones de agregación
```

### Orden de ejecución de SQL (importante saberlo)

```
1. FROM     → se identifica la tabla fuente
2. WHERE    → se filtran las filas individuales
3. GROUP BY → se agrupan las filas restantes
4. HAVING   → se filtran los grupos
5. SELECT   → se calculan y proyectan las columnas del resultado
6. ORDER BY → se ordena el resultado final
7. LIMIT    → se limita la cantidad de filas
```

In [ ]:
# ── Funciones de agregación SIN GROUP BY ─────────────────────────────────────
#
# Sin GROUP BY, las funciones operan sobre TODA la tabla
# El resultado es siempre una sola fila con los valores globales
# Útil para obtener estadísticas generales del dataset rápidamente

print('Estadísticas generales de la tabla clientes:')  # Mostramos un título para el resumen.
# Ejecutamos la consulta con funciones de agregación.
cursor.execute("""
    SELECT
        COUNT(*)           AS total_clientes,     -- cuenta todas las filas
        COUNT(activo)      AS con_datos_activo,   -- cuenta filas donde activo no es NULL
        SUM(activo)        AS clientes_activos,   -- suma de 1s = cuántos están activos
        ROUND(AVG(edad),1) AS edad_promedio,      -- promedio redondeado a 1 decimal
        MIN(edad)          AS cliente_mas_joven,  -- mínimo de la columna edad
        MAX(edad)          AS cliente_mayor       -- máximo de la columna edad
    FROM clientes
""")
filas = cursor.fetchall()  # Traemos la fila única con las estadísticas generales.

for fila in filas:  # Recorremos el resultado aunque solo sea una fila.
    print(fila)  # Imprimimos el resumen estadístico.

In [ ]:
# ── GROUP BY con una columna ──────────────────────────────────────────────────
#
# GROUP BY ciudad agrupa todos los clientes por ciudad
# Luego COUNT(*) cuenta cuántos hay en cada grupo
# El resultado tiene UNA FILA POR CADA CIUDAD DISTINTA
#
# Pregunta de negocio: ¿Cuántos clientes tenemos por ciudad?

# Ejecutamos la consulta agrupando por ciudad.
cursor.execute("""
    SELECT
        ciudad,                      -- columna de agrupación (debe ir en GROUP BY)
        COUNT(*)  AS num_clientes,   -- función de agregación: cuenta filas por grupo
        AVG(edad) AS edad_promedio   -- función de agregación: promedio por grupo
    FROM   clientes
    GROUP  BY ciudad                 -- cada valor único de ciudad forma un grupo
    ORDER  BY num_clientes DESC      -- ordenamos por la cantidad, mayor primero
""")
filas = cursor.fetchall()  # Guardamos una fila por cada ciudad distinta.

for fila in filas:  # Recorremos cada ciudad agrupada.
    print(fila)  # Imprimimos ciudad, número de clientes y edad promedio.

In [ ]:
# ── WHERE + GROUP BY: filtrar ANTES de agrupar ───────────────────────────────
#
# Primero WHERE elimina los clientes inactivos (activo = 1)
# Luego GROUP BY agrupa los que quedaron por categoría
# Esto es correcto: WHERE filtra filas individuales antes de agregar
#
# Pregunta de negocio: distribución por segmento, solo clientes activos

# Ejecutamos la consulta filtrando primero y agrupando después.
cursor.execute("""
    SELECT
        categoria        AS segmento,
        COUNT(*)         AS num_clientes,
        ROUND(AVG(edad)) AS edad_prom
    FROM   clientes
    WHERE  activo = 1           -- primero filtramos: solo activos
    GROUP  BY categoria         -- luego agrupamos los activos por categoría
    ORDER  BY num_clientes DESC
""")
filas = cursor.fetchall()  # Guardamos los grupos de clientes activos por segmento.

for fila in filas:  # Recorremos cada segmento agrupado.
    print(fila)  # Imprimimos segmento, cantidad y edad promedio.

In [ ]:
# ── HAVING: filtrar grupos DESPUÉS de agregar ────────────────────────────────
#
# HAVING es como un WHERE pero para grupos
# Se evalúa DESPUÉS de que GROUP BY ha creado los grupos y calculado las agregaciones
#
# ERROR COMÚN: usar WHERE con funciones de agregación → SQL lanza error
# CORRECTO:    WHERE para columnas normales, HAVING para resultados de agregaciones
#
# Pregunta de negocio: ¿Qué ciudades tienen MÁS DE 2 clientes activos?

# Ejecutamos la consulta agrupando y filtrando grupos con HAVING.
cursor.execute("""
    SELECT
        ciudad,
        COUNT(*) AS num_clientes
    FROM   clientes
    WHERE  activo = 1             -- WHERE filtra filas antes de agrupar
    GROUP  BY ciudad
    HAVING COUNT(*) > 2           -- HAVING filtra grupos después de agregar
    ORDER  BY num_clientes DESC
""")
filas = cursor.fetchall()  # Guardamos las ciudades que pasan el filtro HAVING.

for fila in filas:  # Recorremos cada ciudad del resultado.
    print(fila)  # Imprimimos ciudad y número de clientes activos.

In [ ]:
# ── Análisis de ventas con GROUP BY y HAVING ─────────────────────────────────
#
# Aquí combinamos: GROUP BY + SUM + COUNT + AVG + HAVING
# Esto es el tipo de consulta que se usa a diario en equipos de analytics
#
# Pregunta de negocio: ¿Qué clientes han gastado más de $500.000 en total?
# (estos son candidatos para ofertas de fidelización)

# Ejecutamos la consulta para resumir ventas por cliente.
cursor.execute("""
    SELECT
        id_cliente,
        COUNT(id_venta)            AS num_compras,      -- cuántas veces compró
        SUM(total)                 AS total_gastado,    -- gasto acumulado
        ROUND(AVG(total), 0)       AS ticket_promedio,  -- gasto promedio por compra
        MIN(total)                 AS compra_minima,    -- su compra más pequeña
        MAX(total)                 AS compra_maxima     -- su compra más grande
    FROM   ventas
    GROUP  BY id_cliente                               -- un grupo por cliente
    HAVING SUM(total) > 500000                         -- solo los de alto valor
    ORDER  BY total_gastado DESC
""")
filas = cursor.fetchall()  # Guardamos los clientes cuyo gasto total supera el umbral.

for fila in filas:  # Recorremos cada cliente de alto valor.
    print(fila)  # Imprimimos sus métricas de compra.

---
## 🔗 BLOQUE 4 — JOINs: Cruzar información entre tablas

### ¿Qué es un JOIN y para qué sirve?

Un **JOIN** combina filas de dos o más tablas basándose en una columna que tienen en común (generalmente la clave primaria y la clave foránea).

Es la operación más poderosa de SQL porque **las bases de datos relacionales existen precisamente para esto**: separar la información en tablas normalizadas y luego combinarlas según se necesite.

Sin JOINs, una base de datos relacional pierde su mayor ventaja.

### Tipos de JOIN más usados

```
Tabla A    Tabla B
  ┌──┐       ┌──┐
  │  │       │  │
  │ ███████████ │  INNER JOIN → solo la intersección (filas que existen en ambas)
  │  │       │  │
  └──┘       └──┘

  ┌──┐       ┌──┐
  │  │       │  │
 ████████████ │  │  LEFT JOIN  → todos de A + coincidencias de B (NULL si no hay)
  │  │       │  │
  └──┘       └──┘
```

En la práctica, el **90% de los JOINs** en proyectos de datos son `INNER JOIN` o `LEFT JOIN`.

### Buenas prácticas con JOINs
- Siempre usa **alias** para las tablas (`clientes AS c`, `ventas AS v`) para acortar el código
- Especifica de qué tabla viene cada columna con el prefijo (`c.nombre`, `v.total`)
- Empieza con la tabla más grande o central y ve añadiendo JOINs

In [ ]:
# ── INNER JOIN entre ventas y clientes ───────────────────────────────────────
#
# JOIN (sin especificar tipo) = INNER JOIN por defecto
# ON especifica la condición de cruce: qué columna conecta las dos tablas
#
# Resultado: solo aparecen las ventas que tienen un cliente válido
# (si hubiera una venta con id_cliente=99 y no existiera ese cliente, no aparece)
#
# Usamos aliases: 'v' para ventas, 'c' para clientes
# Prefijamos columnas con el alias: v.total, c.nombre

# Ejecutamos la consulta con INNER JOIN entre ventas y clientes.
cursor.execute("""
    SELECT
        v.id_venta,
        c.nombre    AS cliente,     -- columna 'nombre' de la tabla clientes (c)
        c.ciudad,
        c.categoria AS segmento,
        v.fecha,
        v.total
    FROM   ventas   AS v            -- tabla principal: alias 'v'
    JOIN   clientes AS c            -- tabla a combinar: alias 'c'
      ON   v.id_cliente = c.id_cliente   -- condición: la clave foránea = clave primaria
    ORDER  BY v.total DESC
    LIMIT  8
""")
filas = cursor.fetchall()  # Guardamos las ventas que sí tienen cliente relacionado.

for fila in filas:  # Recorremos cada venta con datos del cliente.
    print(fila)  # Imprimimos venta, cliente, ciudad, segmento, fecha y total.

In [ ]:
# ── JOIN de 3 tablas ──────────────────────────────────────────────────────────
#
# Podemos encadenar múltiples JOINs para cruzar más de dos tablas
# Cada JOIN adicional sigue la misma estructura: JOIN tabla ON condición
#
# Aquí cruzamos las 3 tablas para ver el detalle completo de cada venta:
# quién compró, qué compró, cuánto pagó

# Ejecutamos la consulta cruzando ventas, clientes y productos.
cursor.execute("""
    SELECT
        c.nombre        AS cliente,
        c.categoria     AS segmento,
        p.nombre        AS producto,
        p.categoria     AS cat_producto,
        p.precio        AS precio_unitario,
        v.cantidad,
        v.total,
        v.fecha
    FROM   ventas    AS v
    JOIN   clientes  AS c  ON v.id_cliente  = c.id_cliente   -- 1er JOIN: ventas ↔ clientes
    JOIN   productos AS p  ON v.id_producto = p.id_producto  -- 2do JOIN: ventas ↔ productos
    ORDER  BY v.total DESC
""")
filas = cursor.fetchall()  # Guardamos el detalle completo de cada venta.

for fila in filas:  # Recorremos cada venta enriquecida con cliente y producto.
    print(fila)  # Imprimimos el detalle completo de la venta.

In [ ]:
# ── LEFT JOIN: todos los clientes, compren o no ──────────────────────────────
#
# LEFT JOIN = "dame TODOS los registros de la tabla izquierda (clientes)"
# "y, si tienen coincidencia en ventas, muestro esos datos también"
# "si NO tienen coincidencia → las columnas de ventas quedan como NULL"
#
# DIFERENCIA con INNER JOIN:
#   INNER JOIN: solo clientes que tienen al menos una venta
#   LEFT JOIN:  todos los clientes, incluso los que nunca compraron
#
# Pregunta de negocio: ¿Qué clientes nunca han comprado? (para campaña de reactivación)

# Ejecutamos la consulta con LEFT JOIN para incluir todos los clientes.
cursor.execute("""
    SELECT
        c.nombre,
        c.categoria,
        c.ciudad,
        COUNT(v.id_venta) AS num_compras   -- COUNT de NULL = 0 automáticamente
    FROM   clientes AS c
    LEFT JOIN ventas AS v ON c.id_cliente = v.id_cliente  -- incluye clientes SIN ventas
    GROUP  BY c.id_cliente, c.nombre, c.categoria, c.ciudad
    ORDER  BY num_compras ASC   -- primero los que menos compraron (o nada)
""")
filas = cursor.fetchall()  # Guardamos todos los clientes con su cantidad de compras.

for fila in filas:  # Recorremos cada cliente, compre o no compre.
    print(fila)  # Imprimimos nombre, categoría, ciudad y número de compras.

In [ ]:
# ── JOIN + GROUP BY + WHERE: combinando todo ─────────────────────────────────
#
# Esta es la estructura de consulta más común en analytics de negocio
# Cruzamos tablas, filtramos lo que no necesitamos, agrupamos y calculamos
#
# Pregunta de negocio: ¿Cuánto han vendido los productos de Tecnología por mes?

# Ejecutamos la consulta combinando JOIN, WHERE y GROUP BY.
cursor.execute("""
    SELECT
        SUBSTR(v.fecha, 1, 7)   AS mes,              -- extraer 'YYYY-MM' de la fecha
        p.nombre                AS producto,
        SUM(v.cantidad)         AS unidades_vendidas,
        SUM(v.total)            AS ingresos
    FROM   ventas    AS v
    JOIN   productos AS p ON v.id_producto = p.id_producto
    WHERE  p.categoria = 'Tecnología'    -- filtrar solo productos de Tecnología
    GROUP  BY SUBSTR(v.fecha, 1, 7), p.id_producto
    ORDER  BY mes, ingresos DESC
""")
filas = cursor.fetchall()  # Guardamos las ventas mensuales de productos de tecnología.

for fila in filas:  # Recorremos cada producto por mes.
    print(fila)  # Imprimimos mes, producto, unidades vendidas e ingresos.

---
# 🏗️ BLOQUE 5 — Ejercicio Final: Reporte Ejecutivo de Ventas

## El contexto

El **Director Comercial** necesita un reporte completo de desempeño para la reunión de junta del lunes. Nos envía este correo:

> *"Necesito entender cómo vamos en el año. Específicamente: quiénes son nuestros mejores clientes, qué categorías de producto están jalando, cómo van los meses, si hay clientes que no están comprando y un resumen por ciudad y segmento. Para el lunes por favor."*

Vamos a responder sus 5 preguntas con SQL.

---

### ❓ Pregunta 1 — ¿Quiénes son nuestros mejores clientes?

Necesitamos cruzar ventas con clientes, agrupar por cliente y calcular métricas de valor. Esto se llama **análisis de valor del cliente (CLV)**.

In [ ]:
# TOP clientes por valor total generado
#
# Usamos JOIN para traer el nombre del cliente (que vive en la tabla clientes)
# GROUP BY por cliente para calcular sus métricas acumuladas
# ORDER BY total_gastado DESC para ver primero los más valiosos

# Ejecutamos la consulta para calcular valor por cliente.
cursor.execute("""
    SELECT
        c.nombre                          AS cliente,
        c.categoria                       AS segmento,
        c.ciudad,
        COUNT(v.id_venta)                 AS num_compras,       -- frecuencia de compra
        SUM(v.total)                      AS total_gastado,     -- valor monetario total
        ROUND(AVG(v.total), 0)            AS ticket_promedio,   -- gasto por compra
        MIN(v.fecha)                      AS primera_compra,    -- cuándo entró
        MAX(v.fecha)                      AS ultima_compra      -- última actividad
    FROM   ventas    AS v
    JOIN   clientes  AS c ON v.id_cliente = c.id_cliente
    GROUP  BY c.id_cliente                    -- un grupo por cliente
    ORDER  BY total_gastado DESC              -- los más valiosos primero
""")
filas = cursor.fetchall()  # Guardamos el ranking de clientes por gasto total.

for fila in filas:  # Recorremos cada cliente del ranking.
    print(fila)  # Imprimimos sus métricas de valor.

### ❓ Pregunta 2 — ¿Qué categorías de producto generan más ingresos?

Análisis de mezcla de producto (product mix). Queremos ver el peso de cada categoría sobre el total de ingresos.

In [ ]:
# Análisis de categorías de producto
#
# La parte más interesante es el cálculo del porcentaje:
# Usamos una subconsulta (SELECT SUM(total) FROM ventas) para obtener el total global
# y dividimos los ingresos de cada categoría entre ese total
# El 100.0 (con decimal) fuerza división de punto flotante (sin él, sería división entera)

# Ejecutamos la consulta para resumir ventas por categoría.
cursor.execute("""
    SELECT
        p.categoria                                        AS categoria,
        COUNT(v.id_venta)                                  AS num_ventas,
        SUM(v.cantidad)                                    AS unidades_vendidas,
        SUM(v.total)                                       AS ingresos_totales,
        ROUND(
            SUM(v.total) * 100.0 /
            (SELECT SUM(total) FROM ventas),               -- subconsulta: total global
        1) AS pct_del_total                                -- porcentaje con 1 decimal
    FROM   ventas    AS v
    JOIN   productos AS p ON v.id_producto = p.id_producto
    GROUP  BY p.categoria
    ORDER  BY ingresos_totales DESC
""")
filas = cursor.fetchall()  # Guardamos una fila por categoría de producto.

for fila in filas:  # Recorremos cada categoría calculada.
    print(fila)  # Imprimimos categoría, ventas, unidades, ingresos y porcentaje.

### ❓ Pregunta 3 — ¿Cómo van los meses? ¿Hay estacionalidad?

In [ ]:
# Análisis de tendencia mensual
#
# SUBSTR(fecha, 1, 7) extrae los primeros 7 caracteres de la fecha
# Si fecha = '2024-01-15', SUBSTR(fecha, 1, 7) = '2024-01'
# Esto nos permite agrupar por mes sin importar el día
#
# Esta es la base de cualquier análisis de series de tiempo en SQL

# Ejecutamos la consulta para agrupar ventas por mes.
cursor.execute("""
    SELECT
        SUBSTR(fecha, 1, 7)          AS mes,
        COUNT(*)                     AS num_transacciones,
        COUNT(DISTINCT id_cliente)   AS clientes_unicos,   -- cuántos clientes distintos compraron
        SUM(total)                   AS ingresos,
        ROUND(AVG(total), 0)         AS ticket_promedio
    FROM   ventas
    GROUP  BY SUBSTR(fecha, 1, 7)    -- agrupar por año-mes
    ORDER  BY mes ASC                -- ordenar cronológicamente
""")
filas = cursor.fetchall()  # Guardamos las métricas mensuales.

for fila in filas:  # Recorremos cada mes del resultado.
    print(fila)  # Imprimimos mes, transacciones, clientes, ingresos y ticket promedio.

### ❓ Pregunta 4 — ¿Hay clientes que no están comprando?

Identificar clientes sin actividad es el primer paso de cualquier estrategia de retención o reactivación.

In [ ]:
# Detectar clientes sin compras (registros huérfanos)
#
# Patrón clásico: LEFT JOIN + HAVING COUNT = 0
# El LEFT JOIN incluye todos los clientes, compren o no
# HAVING COUNT(v.id_venta) = 0 filtra solo los que no tienen ninguna venta
#
# COALESCE(valor, alternativa): si el valor es NULL, retorna la alternativa
# Útil con LEFT JOIN porque los campos de la tabla derecha pueden ser NULL

# Ejecutamos la consulta para encontrar clientes sin compras.
cursor.execute("""
    SELECT
        c.nombre,
        c.categoria          AS segmento,
        c.ciudad,
        c.activo,
        COUNT(v.id_venta)    AS total_compras   -- será 0 para los que no compraron
    FROM   clientes AS c
    LEFT JOIN ventas AS v ON c.id_cliente = v.id_cliente
    GROUP  BY c.id_cliente
    HAVING COUNT(v.id_venta) = 0    -- solo los que tienen CERO compras registradas
    ORDER  BY c.categoria           -- ordenar por segmento para priorizar VIPs sin compra
""")
filas = cursor.fetchall()  # Guardamos los clientes que no tienen compras.

for fila in filas:  # Recorremos cada cliente sin actividad.
    print(fila)  # Imprimimos nombre, segmento, ciudad, estado y total de compras.

### ❓ Pregunta 5 — Resumen ejecutivo por ciudad y segmento

La consulta más completa: combina todo lo visto en la clase para generar un reporte ejecutivo listo para presentar.

In [ ]:
# Reporte ejecutivo completo: ciudad × segmento de cliente
#
# Esta consulta integra todo lo aprendido:
#   - LEFT JOIN para incluir clientes sin ventas
#   - WHERE para filtrar solo activos
#   - GROUP BY doble para agrupar por ciudad y segmento
#   - COUNT DISTINCT para contar clientes únicos
#   - COALESCE para manejar NULLs del LEFT JOIN
#   - ROUND para presentación limpia de decimales
#   - ORDER BY múltiple para organizar el reporte lógicamente

# Ejecutamos la consulta ejecutiva por ciudad y segmento.
cursor.execute("""
    SELECT
        c.ciudad,
        c.categoria                             AS segmento,
        COUNT(DISTINCT c.id_cliente)            AS num_clientes,      -- clientes en ese grupo
        COUNT(v.id_venta)                       AS total_ventas,      -- transacciones totales
        COALESCE(SUM(v.total), 0)               AS ingresos,          -- NULL → 0 si no hay ventas
        ROUND(COALESCE(AVG(v.total), 0), 0)     AS ticket_promedio    -- NULL → 0 si no hay ventas
    FROM   clientes AS c
    LEFT JOIN ventas AS v ON c.id_cliente = v.id_cliente
    WHERE  c.activo = 1                         -- solo clientes activos
    GROUP  BY c.ciudad, c.categoria             -- agrupación doble
    ORDER  BY c.ciudad, ingresos DESC           -- ordenar: primero por ciudad, luego por ingresos
""")
filas = cursor.fetchall()  # Guardamos el resumen por ciudad y segmento.

for fila in filas:  # Recorremos cada grupo del reporte.
    print(fila)  # Imprimimos ciudad, segmento, clientes, ventas, ingresos y ticket promedio.

In [ ]:
# ── Bonus: imprimir el reporte final con sqlite3 puro ───────────────────────
#
# En un entorno real, este reporte podría enviarse a un dashboard o a un correo.
# En esta clase lo imprimimos directamente usando sqlite3, cursor.fetchall() y for.

# Ejecutamos la consulta final directamente con el cursor.
cursor.execute("""
    SELECT
        c.nombre                       AS cliente,
        c.ciudad,
        c.categoria                    AS segmento,
        COUNT(v.id_venta)              AS num_compras,
        COALESCE(SUM(v.total), 0)      AS total_gastado,
        COALESCE(MAX(v.fecha), 'Sin compras') AS ultima_compra
    FROM   clientes AS c
    LEFT JOIN ventas AS v ON c.id_cliente = v.id_cliente
    GROUP  BY c.id_cliente
    ORDER  BY total_gastado DESC
""")
reporte_final = cursor.fetchall()  # Guardamos todas las filas del reporte final.

print('═' * 75)  # Imprimimos una línea decorativa para separar el reporte.
print('         REPORTE EJECUTIVO DE CLIENTES - Q1 2024')  # Imprimimos el título del reporte.
print('═' * 75)  # Imprimimos otra línea decorativa debajo del título.

for cliente, ciudad, segmento, num_compras, total_gastado, ultima_compra in reporte_final:  # Recorremos cada tupla de fetchall().
    print(f'Cliente: {cliente}')  # Imprimimos el nombre del cliente.
    print(f'Ciudad: {ciudad}')  # Imprimimos la ciudad del cliente.
    print(f'Segmento: {segmento}')  # Imprimimos la categoría o segmento del cliente.
    print(f'Número de compras: {num_compras}')  # Imprimimos cuántas compras hizo.
    print(f'Total gastado: ${total_gastado:,.0f}')  # Imprimimos el total gastado con formato de moneda.
    print(f'Última compra: {ultima_compra}')  # Imprimimos la fecha de la última compra o "Sin compras".
    print('-' * 75)  # Separamos visualmente un cliente del siguiente.

total_clientes = len(reporte_final)  # Contamos cuántas filas/clientes llegaron en el resultado.
total_ingresos = sum(fila[4] for fila in reporte_final)  # Sumamos la columna total_gastado de cada fila.

print('═' * 75)  # Imprimimos una línea final para cerrar el reporte.
print(f'Total clientes: {total_clientes}')  # Imprimimos el total de clientes reportados.
print(f'Total ingresos: ${total_ingresos:,.0f}')  # Imprimimos el total de ingresos calculado con Python.

---
# 📌 Resumen de la Clase

## Comandos aprendidos hoy

| Comando | ¿Para qué sirve? | Equivalente en pandas |
|---------|-----------------|----------------------|
| `SELECT col AS alias` | Elegir y renombrar columnas | `df[['col']].rename()` |
| `DISTINCT` | Valores únicos | `df['col'].unique()` |
| `ORDER BY col DESC` | Ordenar resultado | `df.sort_values()` |
| `LIMIT n` | Limitar filas | `df.head(n)` |
| `WHERE` | Filtrar filas | `df[df['col'] == valor]` |
| `BETWEEN a AND b` | Rango inclusivo | `df.query('a <= col <= b')` |
| `IN (v1, v2)` | Lista de valores | `df['col'].isin([v1, v2])` |
| `LIKE '%texto%'` | Búsqueda de patrón | `df['col'].str.contains()` |
| `AND / OR / NOT` | Lógica booleana | `& / \| / ~` |
| `GROUP BY` | Agrupar filas | `df.groupby()` |
| `COUNT / SUM / AVG / MIN / MAX` | Funciones de agregación | `.count() / .sum()` etc. |
| `HAVING` | Filtrar grupos | `.groupby().filter()` |
| `INNER JOIN` | Cruzar tablas (solo coincidencias) | `pd.merge(how='inner')` |
| `LEFT JOIN` | Cruzar (todos de la izquierda) | `pd.merge(how='left')` |
| `COALESCE(val, default)` | Reemplazar NULLs | `df.fillna()` |
| `SUBSTR(col, inicio, largo)` | Extraer parte de texto | `df['col'].str[0:7]` |

---

## 🧠 Reglas de oro que siempre debes recordar

1. **El orden de ejecución de SQL** no es el orden en que escribes las cláusulas: `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`
2. **WHERE vs HAVING:** WHERE filtra filas (antes de agrupar), HAVING filtra grupos (después)
3. **La regla del GROUP BY:** toda columna en SELECT que no sea una agregación debe estar en GROUP BY
4. **Usa aliases siempre:** hacen el código más legible y evitan ambigüedades en JOINs
5. **LIMIT en exploración:** nunca hagas `SELECT *` sin LIMIT en tablas de producción

---

## 📝 Tarea para la próxima clase

Responde estas preguntas con SQL sobre nuestra base de datos:

1. ✅ ¿Cuál es el precio promedio por categoría de producto? ¿Cuál categoría tiene el precio más alto?
2. ✅ ¿Qué clientes compraron en más de un mes distinto? (pista: `COUNT(DISTINCT SUBSTR(fecha,1,7))`)
3. ✅ ¿Cuál es el producto más vendido en unidades totales?
4. 🚀 **Extra:** ¿Cuál es el ingreso total por ciudad, incluyendo solo las ciudades con más de $1.000.000 en ventas?

---
*Dev Senior Code · Master IA · Módulo 1 · Clase 3 — SQL para Data Discovery*